
# Lab (Databricks Edition)

## Automate Credit Risk Assessment Test Case Generation from Jira Stories using Databricks Model Serving

**Flow:** Widgets + Jira API token → Jira REST API → Requirement extractor → LLM (Foundation Model APIs) → Markdown parser → Delta table + CSV/JSON in a Unity Catalog Volume.

**Runtime:** Databricks Runtime 15.4 LTS or above. No `%pip install` is required. No model download or GPU is required.

## Before you run

Create a Jira API token and paste it into the `JIRA_API_TOKEN` variable in the Configuration cell.

You can create a Jira API token from your Atlassian account:
[Create an Atlassian API token](https://id.atlassian.com/manage-profile/security/api-tokens)

> **Note:** The API token is used only for this lab. Do not commit the notebook or token to a public repository.

## 0. Unity Catalog objects (adjust names to your permissions)

In [0]:
%sql
CREATE CATALOG IF NOT EXISTS credit_lab;
CREATE SCHEMA IF NOT EXISTS credit_lab.lab;
CREATE VOLUME IF NOT EXISTS credit_lab.lab.output;

## 1. Configuration

In [0]:
# Replace the placeholders with your Jira details and the issue key you want to test.
dbutils.widgets.text("jira_base_url", "https://<your-jira-domain>.atlassian.net")
dbutils.widgets.text("jira_email", "<your-email>")
dbutils.widgets.text("issue_key", "MDP-<issue-number>")
dbutils.widgets.text("endpoint_name", "databricks-meta-llama-3-3-70b-instruct")
dbutils.widgets.text("catalog", "credit_lab")
dbutils.widgets.text("schema", "lab")
dbutils.widgets.text("volume", "output")

JIRA_BASE_URL = dbutils.widgets.get("jira_base_url").strip().rstrip("/")
JIRA_EMAIL    = dbutils.widgets.get("jira_email").strip()
ISSUE_KEY     = dbutils.widgets.get("issue_key").strip()
ENDPOINT_NAME = dbutils.widgets.get("endpoint_name").strip()
CATALOG       = dbutils.widgets.get("catalog").strip()
SCHEMA_NAME   = dbutils.widgets.get("schema").strip()
VOLUME        = dbutils.widgets.get("volume").strip()

# Hardcoded Jira API token (paste yours here; do not share or commit this notebook)
JIRA_API_TOKEN = ""

assert JIRA_BASE_URL and JIRA_EMAIL and ISSUE_KEY, "Set the jira_base_url, jira_email, and issue_key widgets."
assert JIRA_API_TOKEN and "PASTE_YOUR" not in JIRA_API_TOKEN, "Paste your Jira API token into JIRA_API_TOKEN."
print(f"Config OK. Issue: {ISSUE_KEY} | Endpoint: {ENDPOINT_NAME}")

Config OK. Issue: MDP-10 | Endpoint: databricks-meta-llama-3-3-70b-instruct


## 2. Jira Connector

In [0]:
import json
from typing import Any, Dict
import requests

def fetch_jira_issue(issue_key: str) -> Dict[str, Any]:
    """Fetch a Jira issue by key via the Jira Cloud v3 REST API."""
    if not issue_key:
        raise ValueError("Set the issue_key widget before fetching an issue.")
    url = f"{JIRA_BASE_URL}/rest/api/3/issue/{issue_key}"
    response = requests.get(url, auth=(JIRA_EMAIL, JIRA_API_TOKEN), timeout=30)
    try:
        response.raise_for_status()
    except requests.HTTPError as error:
        # Never print response bodies here, they may contain sensitive data.
        print(f"Jira API error: {error} (HTTP {response.status_code})")
        raise
    return response.json()

# Smoke test
test_issue = fetch_jira_issue(ISSUE_KEY)
print(f"Fetched {ISSUE_KEY}. Type: {test_issue['fields']['issuetype']['name']}")

Fetched MDP-10. Type: Idea


## 3. Requirement Extractor

In [0]:
def normalize_field(field_value: Any) -> str:
    """Convert a Jira field, including nested rich-text JSON, into a prompt-friendly string."""
    if field_value is None:
        return ""
    if isinstance(field_value, str):
        return field_value
    return json.dumps(field_value, indent=2, ensure_ascii=False)

def extract_requirements(issue_json: Dict[str, Any]) -> Dict[str, str]:
    """Extract the story summary and description."""
    fields = issue_json.get("fields", {})
    return {
        "summary": normalize_field(fields.get("summary", "")),
        "description": normalize_field(fields.get("description", "")),
    }

reqs = extract_requirements(test_issue)
print("Summary:", reqs["summary"])
print("Description (truncated):\n", reqs["description"][:500])

Summary: Automate Credit Risk Assessment for New Corporate Customers
Description (truncated):
 {
  "type": "doc",
  "version": 1,
  "content": [
    {
      "type": "codeBlock",
      "attrs": {
        "language": "markdown",
        "localId": "f1e81a9a-bdfc-4f42-9920-fa8b5d052f19",
        "wrap": true
      },
      "content": [
        {
          "type": "text",
          "text": "**User Story**  \nAs a credit analyst, I want the system to assess the credit risk of a new corporate customer using registration documents, financial information, external risk data, and industry risk so 


## 4. LLM Setup: Databricks Model Serving (zero download)
Foundation Model APIs are serverless. The SDK authenticates automatically inside the notebook.

In [0]:
from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage, ChatMessageRole

w = WorkspaceClient()

# Optional: list what's available in your workspace/region
for e in w.serving_endpoints.list():
    if e.name.startswith("databricks-"):
        print(e.name)

def generate_with_llm(system_prompt: str, user_prompt: str, max_tokens: int = 4000) -> str:
    """Call a Databricks serving endpoint (foundation model) and return the text."""
    resp = w.serving_endpoints.query(
        name=ENDPOINT_NAME,
        messages=[
            ChatMessage(role=ChatMessageRole.SYSTEM, content=system_prompt),
            ChatMessage(role=ChatMessageRole.USER, content=user_prompt),
        ],
        max_tokens=max_tokens,
        temperature=0.0,   # deterministic, per the story's NFRs
    )
    return resp.choices[0].message.content.strip()

databricks-claude-opus-4-8
databricks-claude-opus-5
databricks-claude-opus-5-5
databricks-gpt-oss-120b
databricks-claude-sonnet-5
databricks-claude-sonnet-5-5
databricks-gpt-oss-20b
databricks-qwen3-next-80b-a3b-instruct
databricks-qwen35-122b-a10b
databricks-llama-4-maverick
databricks-gemma-3-12b
databricks-gte-large-en
databricks-bge-large-en
databricks-meta-llama-3-1-8b-instruct
databricks-meta-llama-3-3-70b-instruct
databricks-qwen3-embedding-0-6b
databricks-openjev-qwen35-4b


> **Model choice:** `databricks-meta-llama-3-3-70b-instruct` is the default. Change the `endpoint_name` widget to try another served model. Never `pip install` a model inside a notebook.

## 5. Prompt Engineering

In [0]:
SYSTEM_PROMPT = "You are a senior QA engineer. You output only Markdown tables, exactly as instructed."

BASE_TESTGEN_PROMPT = """
Read the Jira story requirements below and generate between 12 and 20 test cases covering:
- Company Registration Certificate and GST Certificate validation
- Company-name matching and MCA status
- Sanctions screening and analyst review
- Two years of financial statements
- Current Ratio, Debt-to-Equity Ratio, and Net Profit Margin calculations
- D&B score and Industry Risk Rating
- Final score, risk category, credit limit, and payment terms
- Auditability, response time, data protection, and validation errors
- Positive, negative, edge, and boundary scenarios

Use the scorecard weights and thresholds exactly as supplied. Do not invent scoring logic.

Include tests for:
- Ratio bands
- D&B scores of 700, 650, and 600
- Every industry matrix category
- Final scores of 49, 50, 80, and 81

IMPORTANT OUTPUT RULES:
- Return ONLY the Markdown table. No explanations, intro text, or conclusions.
- Every data row must contain exactly 8 columns.
- The Type value must be exactly one of: Positive, Negative, Edge.
- Do not use the pipe character (|) inside any cell content. Use commas, semicolons, or words instead.
- Number the test cases TC-001, TC-002, and so on.

Format the answer as a Markdown table with exactly these columns:

| ID | Title | Type (Positive/Negative/Edge) | Pre-conditions | Steps | Expected Result | Mapped Requirement(s) | Priority |

Jira story requirements:

Summary:
{summary}

Description:
{description}
"""

def build_testcase_prompt(requirements: Dict[str, str]) -> str:
    # Use replace (not .format) so braces in Jira JSON can never break the template
    return (BASE_TESTGEN_PROMPT
            .replace("{summary}", requirements["summary"])
            .replace("{description}", requirements["description"]))

## 6. The Test Generation Agent

In [0]:
class JiraTestGenerationAgent:
    def __init__(self, issue_key: str):
        self.issue_key = issue_key

    def run(self) -> str:
        issue_json = fetch_jira_issue(self.issue_key)
        requirements = extract_requirements(issue_json)
        prompt = build_testcase_prompt(requirements)

        print("Prompt sent to the model (truncated):")
        print(prompt[:1000], "...\n")

        output = generate_with_llm(SYSTEM_PROMPT, prompt)

        # Truncation guard: 12+ cases won't fit in a short completion
        n_cases = output.count("TC-")
        if n_cases < 12:
            print(f"WARNING: only {n_cases} test cases detected. "
                  "Output may be truncated, re-run with a higher max_tokens.")
        return output

## 7. Run the Agent

In [0]:
agent = JiraTestGenerationAgent(issue_key=ISSUE_KEY)
testcases_markdown = agent.run()
print(testcases_markdown)

Prompt sent to the model (truncated):

Read the Jira story requirements below and generate between 12 and 20 test cases covering:
- Company Registration Certificate and GST Certificate validation
- Company-name matching and MCA status
- Sanctions screening and analyst review
- Two years of financial statements
- Current Ratio, Debt-to-Equity Ratio, and Net Profit Margin calculations
- D&B score and Industry Risk Rating
- Final score, risk category, credit limit, and payment terms
- Auditability, response time, data protection, and validation errors
- Positive, negative, edge, and boundary scenarios

Use the scorecard weights and thresholds exactly as supplied. Do not invent scoring logic.

Include tests for:
- Ratio bands
- D&B scores of 700, 650, and 600
- Every industry matrix category
- Final scores of 49, 50, 80, and 81

IMPORTANT OUTPUT RULES:
- Return ONLY the Markdown table. No explanations, intro text, or conclusions.
- Every data row must contain exactly 8 columns.
- The Type 

## 8. Parse → Delta table + CSV/JSON in a Volume
Delta does not allow spaces or parentheses in column names by default, so the parsed columns are mapped to snake_case names before saving.

In [0]:
import csv, io
from datetime import datetime, timezone

# ---------- 1. Parse the Markdown table ----------
raw = testcases_markdown.strip()
if raw.startswith("```"):                      # strip code fences if present
    raw = raw.strip("`").lstrip("markdown").strip()

table_lines = []
for line in raw.splitlines():
    line = line.strip()
    if "|" not in line:
        continue
    cells = [c.strip() for c in line.strip("|").split("|")]
    if cells and all(c.replace("-", "").replace(":", "").strip() == "" for c in cells):
        continue                               # skip separator rows
    table_lines.append(line)

assert len(table_lines) >= 2, (
    "No Markdown table detected. The model may not have followed the format. "
    "Re-run the agent, or switch to a stronger endpoint."
)

headers = [c.strip() for c in table_lines[0].strip("|").split("|")]
assert len(headers) == 8, f"Expected 8 columns, got {len(headers)}: {headers}"

rows = []
for row_line in table_lines[1:]:
    cells = [c.strip() for c in row_line.strip("|").split("|")]
    cells = (cells + [""] * len(headers))[: len(headers)]   # pad/trim to 8
    rows.append(dict(zip(headers, cells)))
print(f"Parsed {len(rows)} test cases. Columns: {headers}")

# ---------- 2. Map to Delta-safe snake_case columns + audit columns ----------
COLUMNS = ["test_id", "title", "test_type", "preconditions",
           "steps", "expected_result", "mapped_requirements", "priority"]
now = datetime.now(timezone.utc).isoformat(timespec="seconds")
records = []
for r in rows:
    vals = list(r.values())            # positional: robust to minor header wording changes
    rec = dict(zip(COLUMNS, vals))
    rec["source_issue"] = ISSUE_KEY
    rec["generated_at"] = now
    records.append(rec)

# ---------- 3. Delta table in Unity Catalog ----------
df = spark.createDataFrame(records)
table_name = f"{CATALOG}.{SCHEMA_NAME}.credit_risk_test_cases"
df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(table_name)
print(f"Delta table written: {table_name}")

# ---------- 4. CSV and JSON into the Volume ----------
vol = f"/Volumes/{CATALOG}/{SCHEMA_NAME}/{VOLUME}"
buf = io.StringIO()
writer = csv.DictWriter(buf, fieldnames=COLUMNS, extrasaction="ignore")
writer.writeheader()
writer.writerows(records)
dbutils.fs.put(f"{vol}/{ISSUE_KEY}_credit_risk_testcases.csv", buf.getvalue(), overwrite=True)
dbutils.fs.put(f"{vol}/{ISSUE_KEY}_credit_risk_testcases.json",
               json.dumps(records, ensure_ascii=False, indent=2), overwrite=True)
print(f"CSV and JSON saved under {vol}/")

display(df)   # review before trusting: LLM output always needs a human pass

Parsed 28 test cases. Columns: ['ID', 'Title', 'Type (Positive/Negative/Edge)', 'Pre-conditions', 'Steps', 'Expected Result', 'Mapped Requirement(s)', 'Priority']
Delta table written: credit_lab.lab.credit_risk_test_cases
Wrote 5195 bytes.
Wrote 12197 bytes.
CSV and JSON saved under /Volumes/credit_lab/lab/output/


expected_result,generated_at,mapped_requirements,preconditions,priority,source_issue,steps,test_id,test_type,title
Assessment continues to next validation step,2026-10-04T16:21:54+00:00,Document Verification,Company Registration Certificate and GST Certificate are uploaded and valid,High,MDP-10,"Upload certificates, start assessment",TC-001,Positive,Valid Company Registration Certificate and GST Certificate
"Assessment is flagged, no final recommendation",2026-10-04T16:21:54+00:00,Document Verification,Company Registration Certificate is missing,High,MDP-10,Start assessment without certificate,TC-002,Negative,Missing Company Registration Certificate
"Assessment is flagged, no final recommendation",2026-10-04T16:21:54+00:00,Document Verification,GST Certificate is invalid,High,MDP-10,"Upload invalid certificate, start assessment",TC-003,Negative,Invalid GST Certificate
Company validation passes,2026-10-04T16:21:54+00:00,Company and Compliance Checks,Company names match across certificates and customer record,Medium,MDP-10,"Upload certificates, start assessment",TC-004,Positive,Company Name Matching
Assessment is flagged for review,2026-10-04T16:21:54+00:00,Company and Compliance Checks,Company names do not match,Medium,MDP-10,"Upload certificates, start assessment",TC-005,Negative,Non-Matching Company Names
Company validation passes,2026-10-04T16:21:54+00:00,Company and Compliance Checks,MCA status is active,Medium,MDP-10,"Upload certificates, start assessment",TC-006,Positive,Active MCA Status
Assessment is flagged for review,2026-10-04T16:21:54+00:00,Company and Compliance Checks,MCA status is not active,Medium,MDP-10,"Upload certificates, start assessment",TC-007,Negative,Inactive MCA Status
Compliance check passes,2026-10-04T16:21:54+00:00,Sanctions Screening,Sanctions screening returns no potential match,High,MDP-10,Start assessment,TC-008,Positive,Sanctions Screening No Match
Assessment is flagged for analyst review,2026-10-04T16:21:54+00:00,Sanctions Screening,Sanctions screening returns a potential match,High,MDP-10,Start assessment,TC-009,Negative,Sanctions Screening Potential Match
"Ratios are calculated, points are awarded",2026-10-04T16:21:54+00:00,Financial Validation and Ratios,Two years of valid financial statements are provided,Medium,MDP-10,"Upload statements, start assessment",TC-010,Positive,Valid Financial Statements


In [0]:
%sql
SELECT * FROM credit_lab.lab.credit_risk_test_cases WHERE test_type = 'Edge'

expected_result,generated_at,mapped_requirements,preconditions,priority,source_issue,steps,test_id,test_type,title
8 Liquidity points are awarded,2026-10-04T16:21:54+00:00,Credit Risk Scorecard,Current Ratio is exactly 1.0,Low,MDP-10,"Upload statements, start assessment",TC-022,Edge,Edge Case Current Ratio 1.0
12 Leverage points are awarded,2026-10-04T16:21:54+00:00,Credit Risk Scorecard,Debt-to-Equity Ratio is exactly 1.0,Low,MDP-10,"Upload statements, start assessment",TC-023,Edge,Edge Case Debt-to-Equity Ratio 1.0
6 Profitability points are awarded,2026-10-04T16:21:54+00:00,Credit Risk Scorecard,Net Profit Margin is exactly 5%,Low,MDP-10,"Upload statements, start assessment",TC-024,Edge,Edge Case Net Profit Margin 5%
5 External Credit Bureau points are awarded,2026-10-04T16:21:54+00:00,Credit Risk Scorecard,D&B score is exactly 600,Low,MDP-10,"Retrieve D&B score, start assessment",TC-025,Edge,Edge Case D&B Score 600


## Extensions
- Save results back to Jira (attach the CSV via the REST API).
- Multi-agent setup: Requirements Agent → Test Designer Agent → Reviewer Agent (Mosaic AI Agent Framework).
- Parse Jira ADF nodes properly instead of `json.dumps`.

## Lab Summary
You built a Databricks-native agent with widgets and a hardcoded token for config, a served foundation model (`temperature=0`, truncation guard), and durable outputs in a Delta table plus CSV/JSON in a Volume.

**Key lesson:** always review generated test cases before promoting them into your test management process.